<a href="https://colab.research.google.com/github/hannawoloszyn/LLMs-class-ss26/blob/main/session2_NLP_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP in Practice  
## Preprocessing, Representation, and Language Modeling

This notebook gives simple examples of three important techniques in Natural Language Processing (NLP):

1. **Text preprocessing**  
   Cleaning and preparing text

2. **Text representation / feature extraction**  
   Converting text into numbers

3. **Language modeling**  
   Modeling word sequences and predicting the next word

In [1]:
!pip install nltk scikit-learn -q

In [2]:
import nltk
import re
import math

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.util import ngrams
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

In [6]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

### **Part 1: Text Preprocessing**

In [7]:
text = "The cats are running quickly in the garden, and the dog is running too."

In [8]:
tokens = word_tokenize(text)
print(f"Tokens: {tokens}")

Tokens: ['The', 'cats', 'are', 'running', 'quickly', 'in', 'the', 'garden', ',', 'and', 'the', 'dog', 'is', 'running', 'too', '.']


In [9]:
lower_tokens = [token.lower() for token in tokens]
print(f"Lowercased tokens: {lower_tokens}")

Lowercased tokens: ['the', 'cats', 'are', 'running', 'quickly', 'in', 'the', 'garden', ',', 'and', 'the', 'dog', 'is', 'running', 'too', '.']


In [10]:
words_only = [token for token in lower_tokens if token.isalpha()]
print(f"Without punctuation: {words_only}")

Without punctuation: ['the', 'cats', 'are', 'running', 'quickly', 'in', 'the', 'garden', 'and', 'the', 'dog', 'is', 'running', 'too']


In [11]:
stop_words = set(stopwords.words('english'))
filtered_words = [word for word in words_only if word not in stop_words]

print(f"After stop-word removal: {filtered_words}")

After stop-word removal: ['cats', 'running', 'quickly', 'garden', 'dog', 'running']


In [12]:
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in filtered_words]

print(f"After stemming: {stemmed_words}")

After stemming: ['cat', 'run', 'quickli', 'garden', 'dog', 'run']


In [13]:
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

print(f"After lemmatization: {lemmatized_words}")

After lemmatization: ['cat', 'running', 'quickly', 'garden', 'dog', 'running']


In [14]:
def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

processed = preprocess_text(text)

print(f"Final preprocessed text: {processed}")

Final preprocessed text: ['cats', 'running', 'quickly', 'garden', 'dog', 'running']


### **Part 2: Text Representation**

In [15]:
documents = [
    "process mining improves process analysis",
    "process discovery improves business analysis"
]

print("Documents:")
for i, doc in enumerate(documents, start=1):
    print(f"D{i}: {doc}")

Documents:
D1: process mining improves process analysis
D2: process discovery improves business analysis


In [16]:
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(documents)

bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["D1", "D2"]
)

print("Bag of Words representation:")
display(bow_df)

Bag of Words representation:


,analysis,business,discovery,improves,mining,process
D1,1,0,0,1,1,2
D2,1,1,1,1,0,1


In [17]:
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

Vocabulary:
['analysis' 'business' 'discovery' 'improves' 'mining' 'process']


In [18]:
all_words = " ".join(documents).split()
word_freq = Counter(all_words)

freq_df = pd.DataFrame(word_freq.items(), columns=["Word", "Frequency"])
freq_df = freq_df.sort_values(by="Frequency", ascending=False)

print("Word frequencies:")
display(freq_df)

Word frequencies:


,Word,Frequency
0,process,3
2,improves,2
3,analysis,2
1,mining,1
4,discovery,1
5,business,1


In [19]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=["D1", "D2"]
)

print("TF-IDF representation:")
display(tfidf_df.round(3))

TF-IDF representation:


,analysis,business,discovery,improves,mining,process
D1,0.354,0.000,0.000,0.354,0.498,0.708
D2,0.379,0.533,0.533,0.379,0.000,0.379


In [20]:
docs = [
    ["process", "mining", "improves", "process", "analysis"],
    ["process", "discovery", "improves", "business", "analysis"]
]

vocab = sorted(set(word for doc in docs for word in doc))
N = len(docs)

print("Vocabulary:", vocab)

# document frequency
df = {}
for word in vocab:
    df[word] = sum(1 for doc in docs if word in doc)

print("\nDocument Frequency (DF):")
for word, value in df.items():
    print(word, ":", value)

Vocabulary: ['analysis', 'business', 'discovery', 'improves', 'mining', 'process']

Document Frequency (DF):
analysis : 2
business : 1
discovery : 1
improves : 2
mining : 1
process : 2


In [21]:
idf = {}
for word in vocab:
    idf[word] = math.log(N / df[word])

print("IDF values:")
for word, value in idf.items():
    print(f"{word}: {value:.3f}")

IDF values:
analysis: 0.000
business: 0.693
discovery: 0.693
improves: 0.000
mining: 0.693
process: 0.000


In [22]:
doc1 = docs[0]
tf_doc1 = Counter(doc1)

print("TF in Document 1:")
for word in vocab:
    print(f"{word}: {tf_doc1[word]}")

TF in Document 1:
analysis: 1
business: 0
discovery: 0
improves: 1
mining: 1
process: 2


In [23]:
print("TF-IDF in Document 1:")
for word in vocab:
    tfidf_value = tf_doc1[word] * idf[word]
    print(f"{word}: {tfidf_value:.3f}")

TF-IDF in Document 1:
analysis: 0.000
business: 0.000
discovery: 0.000
improves: 0.000
mining: 0.693
process: 0.000


### **Part 3: Language Modeling**

In [24]:
corpus = [
    "I like NLP",
    "I like machine learning",
    "I enjoy NLP",
    "NLP is interesting"
]

print("Training corpus:")
for sentence in corpus:
    print(sentence)

Training corpus:
I like NLP
I like machine learning
I enjoy NLP
NLP is interesting


In [25]:
tokenized_corpus = [sentence.lower().split() for sentence in corpus]
print(tokenized_corpus)

[['i', 'like', 'nlp'], ['i', 'like', 'machine', 'learning'], ['i', 'enjoy', 'nlp'], ['nlp', 'is', 'interesting']]


In [26]:
bigrams_list = []

for sentence in tokenized_corpus:
    bigrams_list.extend(list(ngrams(sentence, 2)))

print("Bigrams:")
print(bigrams_list)

Bigrams:
[('i', 'like'), ('like', 'nlp'), ('i', 'like'), ('like', 'machine'), ('machine', 'learning'), ('i', 'enjoy'), ('enjoy', 'nlp'), ('nlp', 'is'), ('is', 'interesting')]


In [27]:
bigram_counts = Counter(bigrams_list)

print("Bigram counts:")
for bigram, count in bigram_counts.items():
    print(bigram, ":", count)

Bigram counts:
('i', 'like') : 2
('like', 'nlp') : 1
('like', 'machine') : 1
('machine', 'learning') : 1
('i', 'enjoy') : 1
('enjoy', 'nlp') : 1
('nlp', 'is') : 1
('is', 'interesting') : 1


In [29]:
next_word_dict = defaultdict(list)

for sentence in tokenized_corpus:
    for w1, w2 in ngrams(sentence, 2):
        next_word_dict[w1].append(w2)

print("Next-word predictions:")
for word, next_words in next_word_dict.items():
    print(word, "->", next_words)

Next-word predictions:
i -> ['like', 'like', 'enjoy']
like -> ['nlp', 'machine']
machine -> ['learning']
enjoy -> ['nlp']
nlp -> ['is']
is -> ['interesting']


In [31]:
def predict_next_word(word):
    candidates = next_word_dict[word.lower()]
    return Counter(candidates).most_common(1)[0][0]

print("Next word after 'i':", predict_next_word("i"))
print("Next word after 'like':", predict_next_word("like"))
print("Next word after 'nlp':", predict_next_word("nlp"))

Next word after 'i': like
Next word after 'like': nlp
Next word after 'nlp': is
